
# 🖼️ Image Captioning — End‑to‑End (Single Notebook)

This notebook combines **data prep, model, training, evaluation, and inference** for an image captioning system in one place. It supports **Flickr8k (default)** and **COCO 2017 (optional)** using a **ResNet encoder** and an **LSTM decoder with attention** (Bahdanau).

> **Tip:** Training a captioning model from scratch can take hours on CPU. Use a GPU runtime when possible.

---
**Contents**
1. Setup & Configuration
2. Data Download/Preparation (Flickr8k or COCO)
3. Vocabulary & Tokenization
4. Dataset, Transforms, & Dataloader
5. Model: Encoder (CNN) & Decoder (LSTM + Attention)
6. Training Loop (Teacher Forcing, Checkpoints)
7. Evaluation (BLEU‑4)
8. Inference (Greedy / optional Beam) & Attention Visualization
9. Utilities (Save/Load, Reproducibility)

---


## 1) Setup & Configuration

In [ ]:

#@title Install (if needed) and Imports
# If running locally and missing packages, uncomment the next lines.
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install nltk pillow matplotlib tqdm

import os
import re
import io
import json
import math
import time
import random
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    nltk_ok = True
except Exception as e:
    print("NLTK not installed; BLEU will be approximated.")
    nltk_ok = False

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Configuration
CFG = {
    'dataset': 'Flickr8k',  # 'Flickr8k' or 'COCO'
    'data_root': './data',  # base folder for datasets

    # Flickr8k expected structure:
    #   ./data/Flickr8k/images/  (all images)
    #   ./data/Flickr8k/captions/Flickr8k.token.txt
    #   ./data/Flickr8k/captions/Flickr_8k.trainImages.txt
    #   ./data/Flickr8k/captions/Flickr_8k.devImages.txt
    #   ./data/Flickr8k/captions/Flickr_8k.testImages.txt

    # COCO expected structure:
    #   ./data/COCO/train2017/*.jpg
    #   ./data/COCO/val2017/*.jpg
    #   ./data/COCO/annotations/captions_train2017.json
    #   ./data/COCO/annotations/captions_val2017.json

    'min_freq': 5,
    'max_len': 20,
    'batch_size': 64,
    'num_workers': 2,

    'encoder_cnn': 'resnet50',
    'embed_dim': 256,
    'hidden_dim': 512,
    'attention_dim': 256,
    'dropout': 0.3,

    'epochs': 2,            # increase for real training
    'lr': 3e-4,
    'clip': 1.0,
    'teacher_forcing': 0.5,

    'save_dir': './checkpoints',
    'exp_name': 'captioning_resnet50_lstm_attn',

    'fp16': False,          # set True for mixed precision if on recent GPUs
}

os.makedirs(CFG['save_dir'], exist_ok=True)


## 2) Data Download / Preparation


**Flickr8k (Default)**
- Download images and annotation files from public mirrors or Kaggle.
- Place files as described in the config cell.

**COCO 2017 (Optional)**
- Download train2017/val2017 images and captions JSONs from the official COCO site.
- Place under the paths indicated in the config.

> The cells below include *optional* helper functions that attempt to download files if you provide URLs. If files are already present, downloading is skipped.


In [ ]:

from urllib.request import urlretrieve
import zipfile

def safe_download(url: str, dst: str):
    dst_path = Path(dst)
    if dst_path.exists():
        print(f"Already exists: {dst_path}")
        return dst
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {url} -> {dst}")
    urlretrieve(url, dst)
    print("Download complete.")
    return dst

# Example (commented out to avoid accidental downloads):
# Flickr8k images (example URL placeholders; please replace with working links you have rights to use)
# safe_download('https://example.com/Flickr8k_Dataset.zip', './data/Flickr8k/Flickr8k_Dataset.zip')
# with zipfile.ZipFile('./data/Flickr8k/Flickr8k_Dataset.zip') as z: z.extractall('./data/Flickr8k')

# For this notebook to run, ensure files are present on disk following the structure in CFG.


## 3) Vocabulary & Tokenization

In [ ]:

SPECIAL_TOKENS = {
    'pad': '<pad>',
    'bos': '<bos>',
    'eos': '<eos>',
    'unk': '<unk>'
}

class Vocabulary:
    def __init__(self, min_freq=5):
        self.min_freq = min_freq
        self.freqs = {}
        self.stoi = {}
        self.itos = []

    @staticmethod
    def tokenize(text: str) -> List[str]:
        text = text.lower()
        # simple regex tokenizer: words and digits
        return re.findall(r"[a-zA-Z]+|\d+|[^\s\w]", text)

    def build(self, sentences: List[str]):
        for s in sentences:
            for tok in self.tokenize(s):
                self.freqs[tok] = self.freqs.get(tok, 0) + 1
        # add specials first
        self.itos = [SPECIAL_TOKENS['pad'], SPECIAL_TOKENS['bos'], SPECIAL_TOKENS['eos'], SPECIAL_TOKENS['unk']]
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        for tok, f in sorted(self.freqs.items(), key=lambda x: (-x[1], x[0])):
            if f >= self.min_freq and tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)
        print(f"Built vocab: size={len(self)} (min_freq={self.min_freq})")

    def __len__(self):
        return len(self.itos)

    def numericalize(self, tokens: List[str]) -> List[int]:
        return [self.stoi.get(t, self.stoi[SPECIAL_TOKENS['unk']]) for t in tokens]

    def denumericalize(self, ids: List[int]) -> List[str]:
        return [self.itos[i] for i in ids]


def save_vocab(vocab: Vocabulary, path: str):
    obj = {
        'min_freq': vocab.min_freq,
        'itos': vocab.itos
    }
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f)


def load_vocab(path: str) -> Vocabulary:
    with open(path, 'r', encoding='utf-8') as f:
        obj = json.load(f)
    v = Vocabulary(min_freq=obj['min_freq'])
    v.itos = obj['itos']
    v.stoi = {tok: i for i, tok in enumerate(v.itos)}
    return v


## 4) Dataset, Transforms, & Dataloader

In [ ]:

class Flickr8kCaptions:
    """Utility to read Flickr8k token & split files."""
    def __init__(self, root: str):
        root = Path(root)
        self.images_dir = root / 'images'
        self.captions_dir = root / 'captions'
        self.token_file = self.captions_dir / 'Flickr8k.token.txt'
        self.train_file = self.captions_dir / 'Flickr_8k.trainImages.txt'
        self.dev_file = self.captions_dir / 'Flickr_8k.devImages.txt'
        self.test_file = self.captions_dir / 'Flickr_8k.testImages.txt'
        assert self.token_file.exists(), f"Missing {self.token_file}"

        self.image2caps = {}
        with open(self.token_file, 'r', encoding='utf-8') as f:
            for line in f:
                # format: image#idx	caption
                parts = line.strip().split('	')
                if len(parts) != 2: continue
                img_id, cap = parts
                img = img_id.split('#')[0]
                self.image2caps.setdefault(img, []).append(cap)

        def read_list(p):
            with open(p, 'r', encoding='utf-8') as f:
                return [x.strip() for x in f if x.strip()]

        self.train_imgs = read_list(self.train_file)
        self.val_imgs = read_list(self.dev_file)
        self.test_imgs = read_list(self.test_file)


class COCOV2:
    """Lightweight COCO captions reader using JSON only (no pycocotools)."""
    def __init__(self, root: str):
        root = Path(root)
        self.train_dir = root / 'train2017'
        self.val_dir = root / 'val2017'
        ann_dir = root / 'annotations'
        with open(ann_dir / 'captions_train2017.json', 'r') as f:
            train_ann = json.load(f)
        with open(ann_dir / 'captions_val2017.json', 'r') as f:
            val_ann = json.load(f)

        def build(ann, img_dir):
            id2file = {img['id']: img['file_name'] for img in ann['images']}
            file2caps = {}
            for a in ann['annotations']:
                fn = id2file[a['image_id']]
                file2caps.setdefault(fn, []).append(a['caption'])
            files = [f for f in file2caps.keys() if (img_dir / f).exists()]
            return img_dir, file2caps, files

        self.train_dir, self.train_caps, self.train_files = build(train_ann, self.train_dir)
        self.val_dir, self.val_caps, self.val_files = build(val_ann, self.val_dir)


class ImageCaptionDataset(Dataset):
    def __init__(self, dataset_name: str, split: str, cfg: Dict, vocab: Vocabulary=None):
        self.dataset_name = dataset_name
        self.split = split
        self.cfg = cfg
        self.vocab = vocab
        self.max_len = cfg['max_len']

        # Build list of (image_path, caption) pairs
        pairs = []
        if dataset_name.lower() == 'flickr8k':
            flickr = Flickr8kCaptions(Path(cfg['data_root']) / 'Flickr8k')
            if split == 'train':
                files = flickr.train_imgs
            elif split == 'val':
                files = flickr.val_imgs
            else:
                files = flickr.test_imgs
            for fn in files:
                for cap in flickr.image2caps.get(fn, []):
                    pairs.append((flickr.images_dir / fn, cap))
        elif dataset_name.lower() == 'coco':
            coco = COCOV2(Path(cfg['data_root']) / 'COCO')
            if split == 'train':
                files = coco.train_files
                caps_map = coco.train_caps
                base = coco.train_dir
            else:
                files = coco.val_files
                caps_map = coco.val_caps
                base = coco.val_dir
            for fn in files:
                for cap in caps_map.get(fn, []):
                    pairs.append((base / fn, cap))
        else:
            raise ValueError('Unknown dataset: ' + dataset_name)

        self.pairs = pairs
        print(f"Loaded {len(self.pairs)} pairs for {dataset_name} [{split}]")

        # Transforms
        if split == 'train':
            self.tfms = transforms.Compose([
                transforms.Resize((356, 356)),
                transforms.RandomCrop((299, 299)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            self.tfms = transforms.Compose([
                transforms.Resize((299, 299)),
                transforms.CenterCrop((299, 299)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path, caption = self.pairs[idx]
        img = Image.open(path).convert('RGB')
        img = self.tfms(img)
        tokens = [SPECIAL_TOKENS['bos']] + self.vocab.tokenize(caption) + [SPECIAL_TOKENS['eos']]
        ids = self.vocab.numericalize(tokens)
        if len(ids) > self.max_len:
            ids = ids[:self.max_len]
            if ids[-1] != self.vocab.stoi[SPECIAL_TOKENS['eos']]:
                ids[-1] = self.vocab.stoi[SPECIAL_TOKENS['eos']]
        length = len(ids)
        return img, torch.tensor(ids, dtype=torch.long), length


def pad_collate(batch):
    imgs, seqs, lens = zip(*batch)
    imgs = torch.stack(imgs, dim=0)
    max_len = max(lens)
    pad_id = 0  # <pad>
    padded = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :len(s)] = s
    lens = torch.tensor(lens, dtype=torch.long)
    return imgs, padded, lens


### Build Vocabulary from Training Captions

In [ ]:

# Build or load vocabulary
vocab_path = Path(CFG['save_dir']) / f"{CFG['dataset'].lower()}_vocab.json"
if vocab_path.exists():
    vocab = load_vocab(str(vocab_path))
    print(f"Loaded vocab from {vocab_path} (size={len(vocab)})")
else:
    # Collect all training captions
    corpus = []
    if CFG['dataset'].lower() == 'flickr8k':
        flickr = Flickr8kCaptions(Path(CFG['data_root']) / 'Flickr8k')
        for fn in flickr.train_imgs:
            corpus.extend(flickr.image2caps.get(fn, []))
    else:
        coco = COCOV2(Path(CFG['data_root']) / 'COCO')
        for fn in coco.train_files:
            corpus.extend(coco.train_caps.get(fn, []))
    vocab = Vocabulary(min_freq=CFG['min_freq'])
    vocab.build(corpus)
    save_vocab(vocab, str(vocab_path))
    print(f"Saved vocab to {vocab_path}")


### DataLoaders

In [ ]:

train_ds = ImageCaptionDataset(CFG['dataset'], 'train', CFG, vocab)
val_ds   = ImageCaptionDataset(CFG['dataset'], 'val',   CFG, vocab)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], collate_fn=pad_collate, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], collate_fn=pad_collate, pin_memory=True)

len(train_loader), len(val_loader)


## 5) Model: Encoder (CNN) & Decoder (LSTM + Attention)

In [ ]:

class EncoderCNN(nn.Module):
    def __init__(self, cnn_name='resnet50', embed_dim=256, train_backbone=False):
        super().__init__()
        if cnn_name == 'resnet50':
            backbone = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
            modules = list(backbone.children())[:-1]  # remove FC
            self.cnn = nn.Sequential(*modules)
            feat_dim = backbone.fc.in_features
        else:
            raise ValueError('Unsupported CNN: ' + cnn_name)
        for p in self.cnn.parameters():
            p.requires_grad = train_backbone
        self.fc = nn.Linear(feat_dim, embed_dim)
        self.bn = nn.BatchNorm1d(embed_dim)

    def forward(self, images):
        # images: (B,3,H,W)
        feats = self.cnn(images)           # (B, C, 1, 1)
        feats = feats.flatten(1)           # (B, C)
        feats = self.fc(feats)             # (B, E)
        feats = self.bn(feats)
        feats = F.relu(feats)
        return feats                        # (B, E)


class BahdanauAttention(nn.Module):
    def __init__(self, enc_dim, dec_hidden, attn_dim):
        super().__init__()
        self.W = nn.Linear(enc_dim, attn_dim)
        self.U = nn.Linear(dec_hidden, attn_dim)
        self.v = nn.Linear(attn_dim, 1)

    def forward(self, enc_out, hidden):
        # enc_out: (B, E); hidden: (B, H)
        # Expand to time steps = 1 (global image feature). For spatial features, supply (B, T, E).
        # Here we use a single vector feature; attention reduces to a gate.
        score = self.v(torch.tanh(self.W(enc_out) + self.U(hidden)))  # (B,1)
        alpha = torch.softmax(score, dim=1)                            # (B,1)
        context = alpha * enc_out                                      # (B,E)
        return context, alpha


class DecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, attn_dim, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attn = BahdanauAttention(embed_dim, hidden_dim, attn_dim)
        self.lstm = nn.LSTMCell(embed_dim + embed_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, enc_feat, captions, lengths, teacher_forcing=0.5):
        # enc_feat: (B,E), captions: (B,T), lengths: (B,)
        B = enc_feat.size(0)
        T = captions.size(1)
        hidden = torch.zeros(B, self.lstm.hidden_size, device=enc_feat.device)
        cell   = torch.zeros(B, self.lstm.hidden_size, device=enc_feat.device)

        outputs = []
        inputs = captions[:,0]  # should be <bos>
        emb_inputs = self.embed(inputs)

        for t in range(1, T):  # predict token t given previous
            context, _ = self.attn(enc_feat, hidden)  # (B,E)
            lstm_in = torch.cat([emb_inputs, context], dim=1)
            hidden, cell = self.lstm(lstm_in, (hidden, cell))
            logits = self.fc(self.dropout(hidden))
            outputs.append(logits.unsqueeze(1))

            # next input
            teacher = (torch.rand(1).item() < teacher_forcing)
            if teacher:
                next_in = captions[:, t]
            else:
                next_in = logits.argmax(dim=-1)
            emb_inputs = self.embed(next_in)

        outputs = torch.cat(outputs, dim=1)  # (B, T-1, V)
        return outputs

    def greedy_decode(self, enc_feat, bos_id, eos_id, max_len=20):
        B = enc_feat.size(0)
        hidden = torch.zeros(B, self.lstm.hidden_size, device=enc_feat.device)
        cell   = torch.zeros(B, self.lstm.hidden_size, device=enc_feat.device)
        inputs = torch.full((B,), bos_id, dtype=torch.long, device=enc_feat.device)
        emb_inputs = self.embed(inputs)
        seq = []
        for t in range(max_len):
            context, _ = self.attn(enc_feat, hidden)
            lstm_in = torch.cat([emb_inputs, context], dim=1)
            hidden, cell = self.lstm(lstm_in, (hidden, cell))
            logits = self.fc(hidden)
            next_in = logits.argmax(dim=-1)
            seq.append(next_in)
            emb_inputs = self.embed(next_in)
        seq = torch.stack(seq, dim=1)  # (B,T)
        # stop at eos if present
        out = []
        for b in range(B):
            tokens = []
            for tok in seq[b].tolist():
                if tok == eos_id: break
                tokens.append(tok)
            out.append(tokens)
        return out


## 6) Training

In [ ]:

# Build models
encoder = EncoderCNN(CFG['encoder_cnn'], CFG['embed_dim']).to(device)
decoder = DecoderWithAttention(
    vocab_size=len(vocab),
    embed_dim=CFG['embed_dim'],
    hidden_dim=CFG['hidden_dim'],
    attn_dim=CFG['attention_dim'],
    dropout=CFG['dropout']
).to(device)

# Optimization
params = list(decoder.parameters()) + [p for p in encoder.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=CFG['lr'])
criterion = nn.CrossEntropyLoss(ignore_index=0)  # pad=0

scaler = torch.cuda.amp.GradScaler(enabled=CFG['fp16'])

best_bleu = 0.0
ckpt_path = Path(CFG['save_dir']) / f"{CFG['exp_name']}.pt"


def train_one_epoch(epoch):
    encoder.train(); decoder.train()
    total_loss = 0.0
    for i, (imgs, caps, lens) in enumerate(train_loader):
        imgs, caps, lens = imgs.to(device), caps.to(device), lens.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=CFG['fp16']):
            enc = encoder(imgs)
            logits = decoder(enc, caps, lens, teacher_forcing=CFG['teacher_forcing'])  # (B,T-1,V)
            # targets exclude <bos>: shift left
            targets = caps[:,1:logits.size(1)+1]
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(params, CFG['clip'])
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        if (i+1) % 50 == 0:
            print(f"epoch {epoch} | step {i+1}/{len(train_loader)} | loss {total_loss/(i+1):.4f}")
    return total_loss / max(1, len(train_loader))


def evaluate_bleu(sample_limit=2000):
    encoder.eval(); decoder.eval()
    refs = []
    hyps = []
    with torch.no_grad():
        count = 0
        for imgs, caps, lens in val_loader:
            imgs = imgs.to(device)
            enc = encoder(imgs)
            outs = decoder.greedy_decode(enc, vocab.stoi[SPECIAL_TOKENS['bos']], vocab.stoi[SPECIAL_TOKENS['eos']], max_len=CFG['max_len'])
            # references: from original captions in the batch's order
            # For evaluation consistency, we rebuild text from caps
            for b in range(len(outs)):
                # build reference from GT caption (single reference per sample in this loader)
                tgt_ids = caps[b].tolist()
                # strip bos/pad/eos
                try:
                    bos_idx = tgt_ids.index(vocab.stoi[SPECIAL_TOKENS['bos']])
                except ValueError:
                    bos_idx = 0
                if vocab.stoi[SPECIAL_TOKENS['eos']] in tgt_ids:
                    eos_idx = tgt_ids.index(vocab.stoi[SPECIAL_TOKENS['eos']])
                else:
                    eos_idx = len(tgt_ids)
                tgt_ids = tgt_ids[bos_idx+1:eos_idx]
                ref = vocab.denumericalize(tgt_ids)
                hyp = vocab.denumericalize(outs[b])
                if ref and hyp:
                    refs.append([ref])
                    hyps.append(hyp)
            count += len(outs)
            if count >= sample_limit:
                break
    if nltk_ok and len(hyps) > 0:
        smoothie = SmoothingFunction().method4
        bleu4 = corpus_bleu(refs, hyps, smoothing_function=smoothie)
        return bleu4
    else:
        # Simple proxy if nltk unavailable
        return 0.0

# Main training loop (short by default)
for epoch in range(1, CFG['epochs']+1):
    train_loss = train_one_epoch(epoch)
    bleu = evaluate_bleu(sample_limit=1000)
    print(f"Epoch {epoch}: loss={train_loss:.4f}, BLEU-4={bleu:.4f}")
    if bleu > best_bleu:
        best_bleu = bleu
        torch.save({'encoder': encoder.state_dict(), 'decoder': decoder.state_dict(), 'cfg': CFG}, ckpt_path)
        print(f"Saved new best to {ckpt_path}")


## 7) Inference & Attention Visualization

In [ ]:

# Load best (if exists)
if ckpt_path.exists():
    state = torch.load(ckpt_path, map_location=device)
    encoder.load_state_dict(state['encoder'])
    decoder.load_state_dict(state['decoder'])
    print(f"Loaded checkpoint from {ckpt_path}")


def generate_caption(image_path: str, show_image=True):
    encoder.eval(); decoder.eval()

    tfm = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.CenterCrop((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    with torch.no_grad():
        img = Image.open(image_path).convert('RGB')
        tensor = tfm(img).unsqueeze(0).to(device)
        enc = encoder(tensor)
        out_ids = decoder.greedy_decode(enc, vocab.stoi[SPECIAL_TOKENS['bos']], vocab.stoi[SPECIAL_TOKENS['eos']], max_len=CFG['max_len'])[0]
        tokens = vocab.denumericalize(out_ids)
    if show_image:
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Predicted: ' + ' '.join(tokens))
        plt.show()
    return tokens

# Example usage (replace with a real image path):
# generate_caption('./data/Flickr8k/images/123456.jpg')


## 8) Utilities & Helpers

In [ ]:

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Encoder params:', count_parameters(encoder))
print('Decoder params:', count_parameters(decoder))

# Save / Load helpers
def save_checkpoint(path, encoder, decoder, cfg):
    torch.save({'encoder': encoder.state_dict(), 'decoder': decoder.state_dict(), 'cfg': cfg}, path)

def load_checkpoint(path, encoder, decoder, device):
    state = torch.load(path, map_location=device)
    encoder.load_state_dict(state['encoder'])
    decoder.load_state_dict(state['decoder'])
    return state.get('cfg', None)



---
### Notes & Next Steps
- The attention in this minimal example uses a **global image feature** (1 vector); for better captions, use a **spatial feature map** (e.g., take features before global average pooling, flatten spatial grid, and attend over `T = H×W` locations).
- Increase `CFG['epochs']`, lower `min_freq`, and enable **fine‑tuning** of the CNN backbone once the decoder starts learning.
- Add **beam search** for stronger decoding.
- Consider label smoothing and scheduled sampling to stabilize training.
- Evaluate with multiple references (the COCO/Flickr datasets provide 5 refs per image) for fair BLEU.

If you want, I can extend this notebook with **spatial attention + visualization** and **beam search**, or wire it to **Weights & Biases** for experiment tracking.
